# Model Training Notebook

This notebook trains and evaluates the backend models used by the Fair Credit Scoring project. It now matches the unified backend workflow, so you can work with `german`, `heloc`, or run the full pipeline for `both` from the same notebook.

## How to use this notebook

1. Run the setup cells first.
2. Set `DATASET` to `"german"` or `"heloc"` for the step-by-step sections.
3. Set `PIPELINE_DATASET` to `"german"`, `"heloc"`, or `"both"` if you want to run the full pipeline cell.
4. Choose a value for `TABNET_EPOCHS`.
5. Run the sections you need.

Use `TABNET_EPOCHS = 5` for a quick smoke test and `TABNET_EPOCHS = 100` for a fuller run.

Important: fairness-aware reweighting is currently configured only for the German Credit dataset. When `DATASET = "heloc"`, the debiasing and fairness sections are skipped honestly.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.clean_headers import clean_german_credit
from src.preprocess import preprocess_data, load_dataset
from src.train_baseline import train_baseline
from src.train_tabnet import train_tabnet_baseline
from src.train_debiased_tabnet import train_debiased_tabnet
from src.evaluate import evaluate_saved_models, generate_model_comparison
from src.generate_shap_artifacts import generate_backend_shap_artifacts
from src.inference import predict_credit, explain_prediction
from src.train_all import main as train_all_main
from src.config import (
    DEFAULT_DATASET,
    MODEL_DIR,
    OUTPUT_DIR,
    SUPPORTED_DATASETS,
    get_dataset_config,
    get_model_paths,
    get_output_paths,
)

print('Project root:', PROJECT_ROOT)
print('Supported datasets:', SUPPORTED_DATASETS)
print('Model dir:', MODEL_DIR)
print('Output dir:', OUTPUT_DIR)

d:\MY WORK\A NEXT GIG\SHAP\fair_credit_scoring_app\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: D:\MY WORK\A NEXT GIG\SHAP\fair_credit_scoring_app
Supported datasets: ('german', 'heloc')
Model dir: D:\MY WORK\A NEXT GIG\SHAP\fair_credit_scoring_app\models
Output dir: D:\MY WORK\A NEXT GIG\SHAP\fair_credit_scoring_app\outputs


In [ ]:
# Choose one dataset for the step-by-step notebook sections.
DATASET = DEFAULT_DATASET  # "german" or "heloc"

# Choose the dataset scope for the optional full pipeline cell.
PIPELINE_DATASET = DATASET  # "german", "heloc", or "both"

# Change this value before training TabNet.
TABNET_EPOCHS = 100

assert DATASET in SUPPORTED_DATASETS, f"Unsupported DATASET: {DATASET}"
assert PIPELINE_DATASET in (*SUPPORTED_DATASETS, 'both'), f"Unsupported PIPELINE_DATASET: {PIPELINE_DATASET}"

os.environ['FAIR_CREDIT_TABNET_EPOCHS'] = str(TABNET_EPOCHS)

dataset_config = get_dataset_config(DATASET)
model_paths = get_model_paths(DATASET)
output_paths = get_output_paths(DATASET)

print('Step-by-step dataset:', DATASET)
print('Full pipeline dataset:', PIPELINE_DATASET)
print('Display name:', dataset_config['display_name'])
print('Data path:', dataset_config['data_path'])
print('Metrics path:', output_paths['metrics'])
print('Model comparison path:', output_paths['model_comparison'])

if DATASET == 'heloc':
    print('Note: HELOC currently skips fairness-aware reweighting because no sensitive attributes are configured in this project.')

## Unified workflow reference

The backend scripts support the same dataset flags from the terminal:

```powershell
.\.conda\python.exe src\preprocess.py --dataset german
.\.conda\python.exe src\preprocess.py --dataset heloc

.\.conda\python.exe src\train_baseline.py --dataset german
.\.conda\python.exe src\train_baseline.py --dataset heloc

.\.conda\python.exe src\train_tabnet.py --dataset german
.\.conda\python.exe src\train_tabnet.py --dataset heloc

.\.conda\python.exe src\train_debiased_tabnet.py --dataset german

.\.conda\python.exe src\train_all.py --dataset german
.\.conda\python.exe src\train_all.py --dataset heloc
.\.conda\python.exe src\train_all.py --dataset both
```

Use `PIPELINE_DATASET = "both"` only for the full pipeline cell below. Keep `DATASET` set to a single dataset for the step-by-step sections.

In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    train_all_main(dataset=PIPELINE_DATASET)
    print(f'Full pipeline completed for: {PIPELINE_DATASET}')
else:
    print('Set RUN_FULL_PIPELINE = True to execute train_all_main(dataset=PIPELINE_DATASET).')

## 1. Load and inspect the selected dataset

In [ ]:
if DATASET == 'german':
    preview_df = clean_german_credit()
else:
    preview_df = load_dataset(DATASET)

preview_df.head()

In [ ]:
df = load_dataset(DATASET)
target_col = dataset_config['target_col']

print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nTarget distribution:')
print(df[target_col].value_counts())

if dataset_config['sensitive_cols']:
    for col in dataset_config['sensitive_cols']:
        print(f'\n{col} distribution:')
        print(df[col].value_counts())
else:
    print('\nNo sensitive attributes are configured for this dataset in the backend.')

## 2. Preprocess the data

In [ ]:
data = preprocess_data(dataset=DATASET, save_preprocessor=True)
print('X_train shape:', data['X_train'].shape)
print('X_test shape:', data['X_test'].shape)
print('Number of processed features:', len(data['feature_names']))
print('\nFirst 15 processed feature names:')
print(data['feature_names'][:15])

In [ ]:
if data['A_train'].empty:
    print('No protected-attribute frame is available for this dataset.')
else:
    display(data['A_train'].head())

## 3. Train the Logistic Regression baseline

In [ ]:
baseline_result = train_baseline(dataset=DATASET)
baseline_result['metrics']

In [ ]:
if baseline_result['fairness']:
    pd.DataFrame(baseline_result['fairness'])
else:
    print('No fairness output was generated for this dataset.')

## 4. Train the baseline TabNet model

In [ ]:
tabnet_result = train_tabnet_baseline(dataset=DATASET, max_epochs=TABNET_EPOCHS, verbose=1)
tabnet_result['metrics']

In [ ]:
if tabnet_result['fairness']:
    pd.DataFrame(tabnet_result['fairness'])
else:
    print('No fairness output was generated for this dataset.')

## 5. Train the fairness-aware reweighted TabNet model

This section runs only for the German Credit dataset because `sex` and `age_group` are configured there as sensitive attributes.

In [ ]:
if DATASET == 'german':
    debiased_result = train_debiased_tabnet(dataset=DATASET, max_epochs=TABNET_EPOCHS, verbose=1)
    debiased_result['metrics']
else:
    debiased_result = None
    print('Skipped: fairness-aware reweighting is only configured for German Credit in this project.')

In [ ]:
if debiased_result and debiased_result['fairness']:
    pd.DataFrame(debiased_result['fairness'])
else:
    print('No debiased fairness output was generated for this dataset.')

## 6. Evaluate saved models and generate comparison outputs

In [ ]:
fairness_results = evaluate_saved_models(dataset=DATASET)
comparison_df = generate_model_comparison(dataset=DATASET)

print('Models evaluated:', list(fairness_results.keys()))
comparison_df

In [ ]:
metrics = json.loads(output_paths['metrics'].read_text(encoding='utf-8'))
print('Metrics JSON:')
display(pd.DataFrame(metrics).T)

if output_paths['fairness_results'].exists():
    fairness_json = json.loads(output_paths['fairness_results'].read_text(encoding='utf-8'))
    print('Fairness JSON keys:')
    print(list(fairness_json.keys()))
else:
    print('No fairness_results.json file exists for this dataset yet.')

## 7. Generate SHAP artifacts

In [ ]:
shap_artifacts = generate_backend_shap_artifacts(dataset=DATASET)
shap_artifacts

In [ ]:
print('Dataset-specific model files:')
for name, path in model_paths.items():
    print('-', name, '=>', path.name, '(exists)' if path.exists() else '(missing)')

print('\nDataset-specific output files:')
for name, path in output_paths.items():
    print('-', name, '=>', path.name, '(exists)' if path.exists() else '(missing)')

## 8. Test backend inference

In [ ]:
if DATASET == 'german':
    applicant = {
        'age': 34,
        'gender': 'Male',
        'employment_status': '1-4 years',
        'credit_history': 'Existing credits paid back duly',
        'loan_amount': 3500,
        'loan_duration': 24,
        'existing_credits': 1,
        'savings_status': 'Moderate',
        'checking_account_status': 'Low balance',
        'installment_rate': 2,
        'housing_status': 'Own',
        'job_type': 'Skilled employee',
        'dependents': 1,
    }
else:
    applicant = data['X_test_raw'].iloc[0].to_dict()

applicant

In [ ]:
prediction = predict_credit(applicant, dataset=DATASET)
prediction

In [ ]:
explanation = explain_prediction(applicant, dataset=DATASET)
display(explanation['values'])
print(explanation['plain_language'])

## Notes

- Use `DATASET = "german"` or `DATASET = "heloc"` for the interactive notebook sections.
- Use `PIPELINE_DATASET = "both"` only with the optional full pipeline cell if you want one command to train both datasets.
- If TabNet training feels slow, reduce `TABNET_EPOCHS` first.
- For report-quality results, rerun with `TABNET_EPOCHS = 100` or higher after you finish smoke testing.
- The notebook uses the same backend modules as the app, so retraining here updates the artifacts used by Streamlit.